# Day 8: Scaled Dot-Product Attention

## Core Theory (Just-in-Time)

Scaled Dot-Product Attention is the foundational mechanism behind the Transformer architecture. Instead of processing tokens sequentially like RNNs, Transformers process all tokens simultaneously using an attention matrix that defines how much each token should "attend" to every other token.

Given a Query matrix $Q$, Key matrix $K$, and Value matrix $V$, the attention output is computed as:

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

**Why scale by $\sqrt{d_k}$?** 
As the dimensionality of the key vectors ($d_k$) grows, the dot products $QK^T$ can become extremely large in magnitude. Large values push the softmax function into regions with near-zero gradients (saturating the softmax), making backpropagation incredibly slow or halting it completely. Scaling down the dot products by $\sqrt{d_k}$ ensures the variance remains close to 1 (assuming $Q$ and $K$ have unit variance), keeping gradients stable.


### AI Security Implications (Production Best Practices)
While attention mechanisms are fundamental to model performance, they also introduce security considerations. In a production environment, user inputs (queries) directly interact with internal state representations (keys/values). This direct mapping can be exploited via **prompt injection** to manipulate attention weights, steering the model to ignore safety constraints or reveal system prompts. Additionally, because attention matrices can become massive, an attacker might craft excessively long inputs to trigger Resource Exhaustion (Denial of Service) via $O(N^2)$ memory spikes. Implementing **fallback mechanisms** (e.g., sequence length truncation, timeout bounds) and **input sanitization** (scrubbing PII before encoding) are critical pre-processing steps before data reaches the attention layers.


### Basic: Core Implementation
This example isolates the core Scaled Dot-Product Attention concept using NumPy with minimal boilerplate.

In [1]:
import numpy as np

def scaled_dot_product_attention(
    q: np.ndarray, 
    k: np.ndarray, 
    v: np.ndarray, 
    mask: np.ndarray | None = None
) -> np.ndarray:
    """
    Computes Scaled Dot-Product Attention.
    
    Args:
        q: Query matrix of shape (..., seq_len_q, d_k).
        k: Key matrix of shape (..., seq_len_k, d_k).
        v: Value matrix of shape (..., seq_len_v, d_v).
        mask: Optional boolean/binary mask matrix of shape (..., seq_len_q, seq_len_k) 
              to be broadcasted. Elements where mask evaluates to True or 1 will be ignored.
              
    Returns:
        np.ndarray: The output context vectors of shape (..., seq_len_q, d_v).
    """
    d_k = q.shape[-1]
    
    # Compute dot products QK^T
    # q is (..., seq_len_q, d_k)
    # k is (..., seq_len_k, d_k) -> transposed to (..., d_k, seq_len_k)
    # scores is (..., seq_len_q, seq_len_k)
    scores = np.matmul(q, np.swapaxes(k, -2, -1))
    
    # Scale by sqrt(d_k)
    scores = scores / np.sqrt(d_k)
    
    # Apply mask if provided
    if mask is not None:
        # We add a large negative number instead of -inf to avoid NaNs if a whole row is masked
        scores = np.where(mask, -1e9, scores)
        
    # Softmax for attention weights
    # Subtract max for numerical stability (doesn't change mathematical softmax output)
    scores_max = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Multiply by values V
    # weights is (..., seq_len_q, seq_len_k)
    # v is (..., seq_len_k, d_v)
    # output is (..., seq_len_q, d_v)
    output = np.matmul(weights, v)
    
    return output

# Demonstration:
np.random.seed(42)
batch_size = 2
seq_len = 3
d_k = 4
d_v = 4

q_sample = np.random.randn(batch_size, seq_len, d_k)
k_sample = np.random.randn(batch_size, seq_len, d_k)
v_sample = np.random.randn(batch_size, seq_len, d_v)

attention_output = scaled_dot_product_attention(q_sample, k_sample, v_sample)
print("Output shape:", attention_output.shape)
print("Output sample:\n", attention_output[0])


Output shape: (2, 3, 4)
Output sample:
 [[-0.50956193  0.05588786  0.81102648  0.69426007]
 [-0.59107637 -0.1303306   0.63239038  0.75161766]
 [-0.24829184 -0.74099788  0.51188314  0.33323852]]


### Medium: Object-Oriented Implementation
This example introduces clean OOP principles and state management to encapsulate the attention logic.


In [2]:
import numpy as np

class AttentionMechanism:
    def __init__(self, d_k: int):
        self.d_k = d_k
        self.scale_factor = np.sqrt(d_k)
        
    def forward(self, q: np.ndarray, k: np.ndarray, v: np.ndarray, mask: np.ndarray = None) -> np.ndarray:
        scores = np.matmul(q, np.swapaxes(k, -2, -1)) / self.scale_factor
        
        if mask is not None:
            scores = np.where(mask, -1e9, scores)
            
        # Stable softmax
        scores_max = np.max(scores, axis=-1, keepdims=True)
        exp_scores = np.exp(scores - scores_max)
        attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
        
        return np.matmul(attention_weights, v)

attn = AttentionMechanism(d_k=4)
output_medium = attn.forward(q_sample, k_sample, v_sample)
print("Medium Output Shape:", output_medium.shape)


Medium Output Shape: (2, 3, 4)


### Advanced: Production-Grade Implementation
This version features strict type hinting, docstrings, error handling, and exact import syntax, preparing you for IDE-less interviews and production environments.


In [3]:
import numpy as np
from typing import Optional, Tuple
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ScaledDotProductAttention:
    """
    Production-grade Scaled Dot-Product Attention mechanism.
    """
    def __init__(self, d_k: int) -> None:
        if d_k <= 0:
            raise ValueError(f"Dimension d_k must be strictly positive, got {d_k}")
        self.d_k = d_k
        self.scale = np.sqrt(self.d_k)

    def __call__(
        self, 
        q: np.ndarray, 
        k: np.ndarray, 
        v: np.ndarray, 
        mask: Optional[np.ndarray] = None
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compute the attention output and weights safely.
        
        Args:
            q: Queries of shape (..., seq_len_q, d_k)
            k: Keys of shape (..., seq_len_k, d_k)
            v: Values of shape (..., seq_len_k, d_v)
            mask: Optional mask of shape (..., seq_len_q, seq_len_k)
            
        Returns:
            Tuple of (Context Output, Attention Weights)
        """
        try:
            # Shape validation (Fallback mechanism)
            if q.shape[-1] != self.d_k or k.shape[-1] != self.d_k:
                raise ValueError("Query and Key last dimensions must match d_k.")
            if k.shape[-2] != v.shape[-2]:
                raise ValueError("Key and Value sequence lengths must match.")
                
            scores = np.matmul(q, np.swapaxes(k, -2, -1)) / self.scale
            
            if mask is not None:
                scores = np.where(mask, -1e9, scores)
                
            # Numerically stable softmax
            scores_max = np.max(scores, axis=-1, keepdims=True)
            exp_scores = np.exp(scores - scores_max)
            weights = exp_scores / (np.sum(exp_scores, axis=-1, keepdims=True) + 1e-10)
            
            output = np.matmul(weights, v)
            return output, weights
            
        except Exception as e:
            logger.error(f"Attention computation failed: {str(e)}")
            raise

prod_attn = ScaledDotProductAttention(d_k=4)
output_adv, weights_adv = prod_attn(q_sample, k_sample, v_sample)
print("Advanced Output Shape:", output_adv.shape)


Advanced Output Shape: (2, 3, 4)


## Common Pitfalls in Production

1.  **Missing the Scaling Factor:** Failing to divide by $\sqrt{d_k}$ leads to unstable gradients and training stagnation.
2.  **$O(N^2)$ Memory Complexity:** Standard attention computes an $N \times N$ matrix, making it computationally and memory expensive for long sequences. Production models often use FlashAttention to optimize this.
3.  **Numerical Instability in Softmax:** Exponentiating large positive values leads to overflow. Always subtract the maximum value along the axis before computing `np.exp`, which mathematically preserves the Softmax but ensures computational stability.
4.  **Incorrect Masking Values:** Adding `-inf` to masked positions works mathematically but can lead to `NaN` errors during backpropagation if a row is completely masked. Often, `-1e9` is preferred.


## Practical Lab / Homework: Causal Masking

In autoregressive models like GPT, a token should not attend to future tokens. This is achieved using a **causal mask** (or look-ahead mask).

**Your Task:**
Create a causal mask for a sequence of length `seq_len = 4` and apply it to the attention mechanism.
A causal mask is a boolean or binary mask where the upper triangle (excluding the diagonal) is set to `True` (or 1) so it gets masked out.


**Video Walkthrough:**
Please record a brief async video (e.g., via Loom) walking through your causal masking design decisions, explaining how the mask prevents information leakage from future tokens.

In [4]:
import numpy as np

seq_len_lab = 4
d_k = 4
d_v = 4
np.random.seed(42)
q_lab = np.random.randn(1, seq_len_lab, d_k)
k_lab = np.random.randn(1, seq_len_lab, d_k)
v_lab = np.random.randn(1, seq_len_lab, d_v)

# Fully working causal mask implementation
causal_mask = np.triu(np.ones((seq_len_lab, seq_len_lab), dtype=bool), k=1)

# We reuse the basic scaled_dot_product_attention function defined earlier
lab_output = scaled_dot_product_attention(q_lab, k_lab, v_lab, mask=causal_mask)

print("Causal Mask:\n", causal_mask)
print("\nLab Output Shape:\n", lab_output.shape)
print("\nLab Output:\n", lab_output)


Causal Mask:
 [[False  True  True  True]
 [False False  True  True]
 [False False False  True]
 [False False False False]]

Lab Output Shape:
 (1, 4, 4)

Lab Output:
 [[[-0.01349722 -1.05771093  0.82254491 -1.22084365]
  [ 0.12691936 -1.62728084 -0.53560038 -0.32559024]
  [ 0.25481287 -0.86626498  0.12197306 -0.66893378]
  [ 0.01370165 -0.91818559 -0.24293849 -0.20166032]]]


## Reference Links
- [Attention Is All You Need (Original Paper)](https://arxiv.org/abs/1706.03762)
- [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/)
- [NumPy Official Documentation](https://numpy.org/doc/stable/)
